# Эксперимент с Llama 3 8B для хоррор-текстов

Чистый Colab-notebook для англоязычной адаптации эксперимента из статьи *When Detection Fails: The Power of Fine-Tuned Models to Generate Human-Like Social Media Text* на корпус хоррор-историй.

Notebook ожидает заранее подготовленный локальный payload:

```text
horror_experiment_payload.zip
├── train_with_descriptions.csv
├── eval_with_descriptions.csv
├── split_metadata.csv
├── corpus_summary.csv
└── corpus_preparation_report.md
```

GPU-часть сравнивает:

1. человеческие horror-фрагменты;
2. фрагменты, сгенерированные базовой `meta-llama/Meta-Llama-3-8B-Instruct`;
3. фрагменты, сгенерированные той же моделью после QLoRA fine-tuning.


## Next quality run settings

This notebook is copied from the fixed successful Llama run and updates the next experiment design: stricter real artifact markers, longer 250-300 word generations, larger smoke eval/train samples, and cliche diagnostics.


## 0. Среда выполнения

Используй Google Colab с включённым GPU: `Runtime -> Change runtime type -> T4 GPU` или лучше.

Llama 3 8B — gated-модель на Hugging Face. Нужно принять лицензию модели и передать `HF_TOKEN` через Colab secrets/environment или раскомментировать `notebook_login()` ниже.


In [ ]:
#@title 1. Установка зависимостей
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece scipy scikit-learn matplotlib tqdm evaluate huggingface_hub
!pip -q install pandas==2.2.2


In [ ]:
#@title 2. Импорты и базовая конфигурация
from pathlib import Path
import os
import re
import math
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

CONFIG = {
    "GEN_MODEL_NAME": "meta-llama/Meta-Llama-3-8B-Instruct",
    "DETECTOR_MODEL_NAME": "openai-community/roberta-base-openai-detector",
    "PAYLOAD_ZIP": "/content/horror_experiment_payload.zip",
    "WORK_DIR": "/content/horror_experiment",
    "SEED": 42,

    # Next quality run: still bounded, but large enough for meaningful diagnostics.
    # Interactive prompts stay enabled so these can be lowered in Colab if needed.
    "INTERACTIVE_CONFIG": True,
    "SMOKE_TEST": True,
    "SMOKE_TRAIN_SAMPLES": 1500,
    "SMOKE_EVAL_SAMPLES": 100,
    "MAX_TRAIN_SAMPLES": 2000,
    "MAX_EVAL_SAMPLES": 500,
    "BASE_GENERATIONS": 100,

    # Match generated fragment length to the human eval chunks.
    "MAX_NEW_TOKENS": 360,
    "TEMPERATURE": 0.9,
    "TOP_P": 0.92,
    "LORA_R": 8,
    "LORA_ALPHA": 16,
    "LORA_DROPOUT": 0.10,
    "NUM_EPOCHS": 1,
    "LEARNING_RATE": 1e-5,
    "BATCH_SIZE": 1,
    "GRAD_ACCUM": 8,
    "MAX_SEQ_LENGTH": 1536,
}

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])

WORK_DIR = Path(CONFIG["WORK_DIR"])
DATA_DIR = WORK_DIR / "data"
OUT_DIR = WORK_DIR / "outputs"
MODEL_DIR = WORK_DIR / "models"
for directory in [DATA_DIR, OUT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Рабочая папка:", WORK_DIR)


In [ ]:
#@title 3. Авторизация Hugging Face для gated Llama
from huggingface_hub import HfApi, login, notebook_login
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Авторизация выполнена через HF token из environment.")
else:
    print("HF_TOKEN не найден. Если загрузка модели упадёт, раскомментируй и запусти notebook_login().")
    # notebook_login()

try:
    HfApi().model_info(CONFIG["GEN_MODEL_NAME"])
    print("Доступ к gated Llama repo подтверждён:", CONFIG["GEN_MODEL_NAME"])
except GatedRepoError as exc:
    raise RuntimeError(
        "Нет доступа к gated repo Llama. Открой страницу "
        f"https://huggingface.co/{CONFIG['GEN_MODEL_NAME']}, нажми Request access / Agree, "
        "дождись подтверждения доступа и используй HF_TOKEN именно от этого аккаунта. "
        "После этого перезапусти runtime Colab."
    ) from exc
except HfHubHTTPError as exc:
    raise RuntimeError(
        "Не удалось проверить доступ к Hugging Face model repo. "
        "Проверь HF_TOKEN и подключение к интернету."
    ) from exc


In [ ]:
#@title 4. Загрузка и распаковка подготовленного payload
from google.colab import files

payload_zip = Path(CONFIG["PAYLOAD_ZIP"])
if not payload_zip.exists():
    print("Загрузи horror_experiment_payload.zip")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Payload не был загружен.")
    first_name = next(iter(uploaded.keys()))
    shutil.move(first_name, payload_zip)

payload_dir = DATA_DIR / "payload"
payload_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(payload_zip, "r") as archive:
    archive.extractall(payload_dir)

print("Файлы payload:")
for path in sorted(payload_dir.iterdir()):
    print("-", path.name)

report_path = payload_dir / "corpus_preparation_report.md"
if report_path.exists():
    print("\nКраткий отчёт из payload:\n")
    print(report_path.read_text(encoding="utf-8")[:2500])


In [ ]:
#@title 5. Загрузка train/eval CSV, аудит корпуса и настройка параметров запуска
full_train_df = pd.read_csv(payload_dir / "train_with_descriptions.csv")
full_eval_df = pd.read_csv(payload_dir / "eval_with_descriptions.csv")
split_df = pd.read_csv(payload_dir / "split_metadata.csv")
summary_df = pd.read_csv(payload_dir / "corpus_summary.csv")

print("Payload загружен:")
print("- train chunks в payload:", len(full_train_df))
print("- eval chunks в payload:", len(full_eval_df))
print("- stories в split metadata:", len(split_df))

# ---------------------------------------------------------------------
# ВАЖНО: ранние smoke-прогоны показали повторяющиеся технические артефакты
# GraphicsUnit / #echo / Compatible / 다운받기 / وینت после fine-tuning.
# Поэтому перед SFT фильтруем target-тексты, но не считаем обычные слова
# FIRST/LAST артефактами: локальная переоценка показала, что они удаляли
# много нормальной прозы.
# ---------------------------------------------------------------------

ARTIFACT_MARKERS = [
    "GraphicsUnit", "#echo", "AnchorStyles", "Compatible", "DOWNLOADING",
    "다운받기", "وینت", "namespace", "href=", "http://", "https://",
    "/**/", "Object#", "gcاسطة", "Hlav",
]

def text_quality_report(text: str) -> dict:
    text = "" if pd.isna(text) else str(text)
    tokens = re.findall(r"\S+", text)
    words = re.findall(r"[A-Za-z][A-Za-z']*", text)
    alpha_chars = [ch for ch in text if ch.isalpha()]
    non_latin_alpha = [
        ch for ch in alpha_chars
        if not ("a" <= ch.lower() <= "z")
    ]
    long_tokens = [tok for tok in tokens if len(tok) > 28]
    lower_text = text.lower()
    markers_found = [m for m in ARTIFACT_MARKERS if m.lower() in lower_text]
    repeated_token_pattern = bool(re.search(r"(\b[A-Za-z_#]{3,}\b)(?:[\W_]+\1){4,}", text))
    code_like_score = sum(text.count(x) for x in ["{", "}", "();", "</", "/>", "==", "::", "\\", "/*", "*/"])
    return {
        "n_chars": len(text),
        "n_tokens": len(tokens),
        "n_words": len(words),
        "avg_word_len": float(np.mean([len(w) for w in words])) if words else 0.0,
        "non_latin_alpha_ratio": len(non_latin_alpha) / max(1, len(alpha_chars)),
        "long_token_count": len(long_tokens),
        "markers_found": "|".join(markers_found),
        "has_artifact_marker": len(markers_found) > 0,
        "repeated_token_pattern": repeated_token_pattern,
        "code_like_score": code_like_score,
    }

def is_clean_horror_text(text: str) -> bool:
    q = text_quality_report(text)
    # Порог специально мягкий: он не удаляет нормальную прозу с тире/кавычками,
    # но удаляет код, мусор, неанглийские строки и сверхдлинные технические токены.
    return (
        80 <= q["n_words"] <= 380
        and 500 <= q["n_chars"] <= 3500
        and 3.0 <= q["avg_word_len"] <= 7.5
        and q["non_latin_alpha_ratio"] <= 0.03
        and q["long_token_count"] == 0
        and not q["has_artifact_marker"]
        and not q["repeated_token_pattern"]
        and q["code_like_score"] <= 8
    )

def audit_and_filter(df: pd.DataFrame, name: str) -> pd.DataFrame:
    quality = df["text"].apply(text_quality_report).apply(pd.Series)
    audited = pd.concat([df.reset_index(drop=True), quality], axis=1)
    audited["is_clean_for_sft"] = audited["text"].apply(is_clean_horror_text)

    print(f"\n{name}:")
    print("- всего строк:", len(audited))
    print("- чистых строк:", int(audited["is_clean_for_sft"].sum()))
    print("- удалено строк:", int((~audited["is_clean_for_sft"]).sum()))

    marker_rows = audited[audited["has_artifact_marker"]]
    if len(marker_rows):
        print(f"- строк с явными artifact markers: {len(marker_rows)}")
        display(marker_rows[["source_file", "markers_found", "text"]].head(5))

    removed_path = OUT_DIR / f"{name}_removed_by_quality_filter.csv"
    audited.loc[~audited["is_clean_for_sft"]].to_csv(removed_path, index=False)
    print("- удалённые строки сохранены:", removed_path)

    clean_df = audited.loc[audited["is_clean_for_sft"], df.columns].reset_index(drop=True)
    if len(clean_df) < 50:
        raise RuntimeError(
            f"После фильтрации {name} осталось слишком мало строк: {len(clean_df)}. "
            "Нужно пересобрать payload или ослабить фильтр вручную после просмотра удалённых строк."
        )
    return clean_df

clean_train_df = audit_and_filter(full_train_df, "train")
clean_eval_df = audit_and_filter(full_eval_df, "eval")

CONFIG["MAX_TRAIN_SAMPLES"] = min(CONFIG["MAX_TRAIN_SAMPLES"], len(clean_train_df))
CONFIG["MAX_EVAL_SAMPLES"] = min(CONFIG["MAX_EVAL_SAMPLES"], len(clean_eval_df))
CONFIG["BASE_GENERATIONS"] = min(CONFIG["BASE_GENERATIONS"], len(clean_eval_df))
CONFIG["SMOKE_TRAIN_SAMPLES"] = min(CONFIG["SMOKE_TRAIN_SAMPLES"], len(clean_train_df))
CONFIG["SMOKE_EVAL_SAMPLES"] = min(CONFIG["SMOKE_EVAL_SAMPLES"], len(clean_eval_df))


def ask_bool(name: str, default: bool) -> bool:
    raw = input(f"{name} [{default}] (y/n, Enter = default): ").strip().lower()
    if raw == "":
        return default
    return raw in {"y", "yes", "true", "1", "да", "д"}


def ask_int(name: str, default: int, min_value: int | None = None, max_value: int | None = None) -> int:
    raw = input(f"{name} [{default}] (Enter = default): ").strip()
    value = default if raw == "" else int(raw)
    if min_value is not None:
        value = max(min_value, value)
    if max_value is not None:
        value = min(max_value, value)
    return value


def ask_float(name: str, default: float, min_value: float | None = None) -> float:
    raw = input(f"{name} [{default}] (Enter = default): ").strip()
    value = default if raw == "" else float(raw)
    if min_value is not None:
        value = max(min_value, value)
    return value


if CONFIG["INTERACTIVE_CONFIG"]:
    print("\nИнтерактивная настройка. Для исправленного smoke-прогона можно нажимать Enter.")
    CONFIG["SMOKE_TEST"] = ask_bool("SMOKE_TEST", CONFIG["SMOKE_TEST"])
    CONFIG["NUM_EPOCHS"] = ask_int("NUM_EPOCHS", CONFIG["NUM_EPOCHS"], min_value=1)
    CONFIG["LORA_R"] = ask_int("LORA_R", CONFIG["LORA_R"], min_value=1)
    CONFIG["LEARNING_RATE"] = ask_float("LEARNING_RATE", CONFIG["LEARNING_RATE"], min_value=1e-7)
    CONFIG["MAX_NEW_TOKENS"] = ask_int("MAX_NEW_TOKENS", CONFIG["MAX_NEW_TOKENS"], min_value=32)

    if CONFIG["SMOKE_TEST"]:
        CONFIG["SMOKE_TRAIN_SAMPLES"] = ask_int(
            "SMOKE_TRAIN_SAMPLES",
            CONFIG["SMOKE_TRAIN_SAMPLES"],
            min_value=1,
            max_value=len(clean_train_df),
        )
        CONFIG["SMOKE_EVAL_SAMPLES"] = ask_int(
            "SMOKE_EVAL_SAMPLES",
            CONFIG["SMOKE_EVAL_SAMPLES"],
            min_value=1,
            max_value=len(clean_eval_df),
        )
    else:
        CONFIG["MAX_TRAIN_SAMPLES"] = ask_int(
            "MAX_TRAIN_SAMPLES",
            CONFIG["MAX_TRAIN_SAMPLES"],
            min_value=1,
            max_value=len(clean_train_df),
        )
        CONFIG["MAX_EVAL_SAMPLES"] = ask_int(
            "MAX_EVAL_SAMPLES",
            CONFIG["MAX_EVAL_SAMPLES"],
            min_value=1,
            max_value=len(clean_eval_df),
        )

if CONFIG["SMOKE_TEST"]:
    train_df = clean_train_df.sample(
        min(CONFIG["SMOKE_TRAIN_SAMPLES"], len(clean_train_df)),
        random_state=CONFIG["SEED"],
    ).reset_index(drop=True)
    eval_df = clean_eval_df.sample(
        min(CONFIG["SMOKE_EVAL_SAMPLES"], len(clean_eval_df)),
        random_state=CONFIG["SEED"],
    ).reset_index(drop=True)
else:
    train_df = clean_train_df.sample(
        min(CONFIG["MAX_TRAIN_SAMPLES"], len(clean_train_df)),
        random_state=CONFIG["SEED"],
    ).reset_index(drop=True)
    eval_df = clean_eval_df.sample(
        min(CONFIG["MAX_EVAL_SAMPLES"], len(clean_eval_df)),
        random_state=CONFIG["SEED"],
    ).reset_index(drop=True)

N_GENERATE = min(CONFIG["BASE_GENERATIONS"], len(eval_df))
if CONFIG["SMOKE_TEST"]:
    N_GENERATE = min(N_GENERATE, CONFIG["SMOKE_EVAL_SAMPLES"])

eval_gen_df = eval_df.head(N_GENERATE).copy()

print("\nИтоговые параметры запуска:")
for key in [
    "SMOKE_TEST",
    "SMOKE_TRAIN_SAMPLES",
    "SMOKE_EVAL_SAMPLES",
    "MAX_TRAIN_SAMPLES",
    "MAX_EVAL_SAMPLES",
    "BASE_GENERATIONS",
    "NUM_EPOCHS",
    "LORA_R",
    "LEARNING_RATE",
    "MAX_NEW_TOKENS",
]:
    print(f"- {key}: {CONFIG[key]}")

print("\nВыборка для текущего запуска:")
print("- train chunks:", len(train_df))
print("- eval chunks:", len(eval_df))
print("- prompts для генерации/оценки:", len(eval_gen_df))
display(train_df.head(2))


In [ ]:
#@title 6. Загрузка базовой Llama 3 8B в 4-bit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("GPU не найден. В Colab выбери Runtime -> Change runtime type -> T4 GPU и перезапусти runtime.")

print("CUDA device:", torch.cuda.get_device_name(0))
print("CUDA memory allocated before load:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["GEN_MODEL_NAME"], use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["GEN_MODEL_NAME"],
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
print("Модель загружена:", CONFIG["GEN_MODEL_NAME"])
print("CUDA memory allocated after load:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


In [ ]:
#@title 7. Вспомогательные функции генерации
def build_messages(description: str):
    return [
        {
            "role": "system",
            "content": (
                "You write literary horror fragments in English. Write naturally, without explanations. "
                "Use concrete sensory detail, grounded scene movement, and varied sentence rhythm. "
                "Avoid generic horror cliches and do not summarize the premise."
            ),
        },
        {
            "role": "user",
            "content": (
                normalize_generation_description(description)
                + "\n\nLength target: 250-300 words. "
                + "Keep the fragment as one continuous scene. "
                + "Avoid phrases like 'shiver down my spine', 'blood ran cold', "
                + "'could not shake the feeling', and 'darkness seemed alive'."
            ),
        },
    ]


def build_prompt(description: str) -> str:
    messages = build_messages(description)
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return "System: " + messages[0]["content"] + "\nUser: " + description + "\nAssistant:"


def clean_generation(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^(Assistant:|assistant:)\s*", "", text).strip()
    return text


def generation_has_artifacts(text: str) -> bool:
    q = text_quality_report(text) if "text_quality_report" in globals() else {}
    return (
        any(marker.lower() in str(text).lower() for marker in ARTIFACT_MARKERS)
        or q.get("non_latin_alpha_ratio", 0) > 0.05
        or q.get("long_token_count", 0) > 0
        or q.get("repeated_token_pattern", False)
        or q.get("avg_word_len", 0) > 8.0
    )


def generate_one(model, description: str) -> str:
    prompt = build_prompt(description)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Для Llama 3 chat-моделей нужно останавливать генерацию не только на eos,
    # но и на <|eot_id|>, иначе модель может продолжать следующий chat-turn.
    terminators = [tokenizer.eos_token_id]
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot_id, int) and eot_id != tokenizer.unk_token_id:
        terminators.append(eot_id)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
            do_sample=True,
            temperature=CONFIG["TEMPERATURE"],
            top_p=CONFIG["TOP_P"],
            repetition_penalty=1.08,
            no_repeat_ngram_size=5,
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return clean_generation(tokenizer.decode(new_tokens, skip_special_tokens=True))


In [ ]:
#@title 8. Генерация base AI horror-фрагментов
base_rows = []
for _, row in tqdm(eval_gen_df.iterrows(), total=len(eval_gen_df)):
    generated = generate_one(base_model, row["description"])
    base_rows.append({
        "source_story_id": row["story_id"],
        "source_file": row["source_file"],
        "description": row["description"],
        "text": generated,
        "label": "ai_base",
        "generator": CONFIG["GEN_MODEL_NAME"],
    })

base_ai_df = pd.DataFrame(base_rows)
base_path = DATA_DIR / "ai_base_generations.csv"
base_ai_df.to_csv(base_path, index=False)
display(base_ai_df.head())
print("Сохранено:", base_path)


In [ ]:
#@title 9. Подготовка SFT-датасета с assistant-only loss
from datasets import Dataset

# Важно: в исправленной версии loss считается только по ответу assistant.
# System/User prompt маскируется значением -100. Это снижает риск того,
# что модель будет учить служебную chat-разметку вместо художественного текста.

def build_sft_texts(description: str, target_text: str) -> tuple[str, str]:
    prompt_messages = [
        {
            "role": "system",
            "content": (
                "You write literary horror fragments in English. Write naturally, without explanations. "
                "Use concrete sensory detail, grounded scene movement, and varied sentence rhythm. "
                "Avoid generic horror cliches and do not summarize the premise."
            ),
        },
        {
            "role": "user",
            "content": (
                normalize_generation_description(description)
                + "\n\nLength target: 250-300 words. "
                + "Keep the fragment as one continuous scene. "
                + "Avoid phrases like 'shiver down my spine', 'blood ran cold', "
                + "'could not shake the feeling', and 'darkness seemed alive'."
            ),
        },
    ]
    full_messages = prompt_messages + [{"role": "assistant", "content": str(target_text)}]

    if getattr(tokenizer, "chat_template", None):
        prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    else:
        prompt_text = f"System: {prompt_messages[0]['content']}\nUser: {description}\nAssistant:"
        full_text = f"{prompt_text} {target_text}{tokenizer.eos_token}"
    return prompt_text, full_text


def tokenize_sft_example(row: dict) -> dict:
    prompt_text, full_text = build_sft_texts(row["description"], row["text"])
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=CONFIG["MAX_SEQ_LENGTH"],
    )["input_ids"]

    # Если из-за truncation ответ полностью пропал, пример лучше отбросить.
    if len(full_ids) <= len(prompt_ids) + 8:
        return {"input_ids": [], "attention_mask": [], "labels": [], "keep": False}

    labels = [-100] * min(len(prompt_ids), len(full_ids)) + full_ids[min(len(prompt_ids), len(full_ids)):]
    labels = labels[:len(full_ids)]

    # Дополнительный контроль: в labels должен быть хотя бы небольшой кусок assistant answer.
    supervised_tokens = sum(1 for x in labels if x != -100)
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
        "keep": supervised_tokens >= 32,
    }


raw_train_dataset = Dataset.from_pandas(train_df[["description", "text"]])
tokenized_train_dataset = raw_train_dataset.map(tokenize_sft_example, remove_columns=raw_train_dataset.column_names)
tokenized_train_dataset = tokenized_train_dataset.filter(lambda x: x["keep"])
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["keep"])

print("SFT-примеры после assistant-only tokenization:", len(tokenized_train_dataset))
if len(tokenized_train_dataset) == 0:
    raise RuntimeError("SFT dataset пуст после токенизации. Проверь формат description/text.")

# Покажем один пример для визуального контроля.
sample_row = train_df.iloc[0]
sample_prompt, sample_full = build_sft_texts(sample_row["description"], sample_row["text"])
print("PROMPT PREVIEW:\n", sample_prompt[:800])
print("\nFULL SFT TEXT PREVIEW:\n", sample_full[:1200])
print("\nSupervised tokens в первом tokenized-примере:",
      sum(1 for x in tokenized_train_dataset[0]["labels"] if x != -100))


In [ ]:
#@title 10. QLoRA fine-tuning через masked labels
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import Trainer, TrainingArguments

# Подготовка k-bit модели к обучению.
model_for_ft = prepare_model_for_kbit_training(base_model)
model_for_ft.config.use_cache = False

# Для диагностического запуска используем только q_proj/v_proj.
# Это делает адаптер менее агрессивным и снижает риск разрушения генерации.
target_modules = ["q_proj", "v_proj"]
lora_config = LoraConfig(
    r=CONFIG["LORA_R"],
    lora_alpha=CONFIG["LORA_ALPHA"],
    target_modules=target_modules,
    lora_dropout=CONFIG["LORA_DROPOUT"],
    bias="none",
    task_type="CAUSAL_LM",
)
model_for_ft = get_peft_model(model_for_ft, lora_config)
model_for_ft.print_trainable_parameters()

class CausalLMCollatorWithLabels:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids, attention_mask, labels = [], [], []
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.tokenizer.pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "horror_llama3_qlora_adapter"),
    num_train_epochs=CONFIG["NUM_EPOCHS"],
    per_device_train_batch_size=CONFIG["BATCH_SIZE"],
    gradient_accumulation_steps=CONFIG["GRAD_ACCUM"],
    learning_rate=CONFIG["LEARNING_RATE"],
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    logging_steps=5,
    save_strategy="no",
    fp16=not use_bf16,
    bf16=use_bf16,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model_for_ft,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    data_collator=CausalLMCollatorWithLabels(tokenizer),
)

train_result = trainer.train()
print("Train result:", train_result)

adapter_path = MODEL_DIR / "horror_llama3_qlora_adapter_final"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print("Адаптер сохранён:", adapter_path)
print("Важно: fine-tuned генерация в следующей ячейке будет идти через заново загруженную base model + adapter.")


In [ ]:
#@title 11. Генерация fine-tuned AI horror-фрагментов через fresh base model + adapter + sanity gate
import gc
from peft import PeftModel

# Освобождаем training model, заново загружаем чистую base model
# и подключаем сохранённый LoRA adapter.
try:
    del trainer
    del model_for_ft
    del base_model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("CUDA memory allocated before fresh reload:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

fresh_base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["GEN_MODEL_NAME"],
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
fresh_base_model.config.use_cache = False

ft_model = PeftModel.from_pretrained(fresh_base_model, adapter_path)
ft_model.eval()
print("Fresh base model + adapter загружены.")
print("CUDA memory allocated after adapter load:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

# Контрольная генерация до запуска полного evaluation.
sanity_prompt = eval_gen_df.iloc[0]["description"]
sanity_text = generate_one(ft_model, sanity_prompt)
print("SANITY PROMPT:\n", sanity_prompt[:1000])
print("\nSANITY GENERATION:\n", sanity_text[:2000])

if generation_has_artifacts(sanity_text):
    raise RuntimeError(
        "Fine-tuned generation всё ещё содержит артефакты. "
        "Остановлено до stylometry/detector. Проверь train_removed_by_quality_filter.csv, "
        "уменьши SMOKE_TRAIN_SAMPLES до 300 или пересобери payload из более чистого корпуса."
    )

ft_rows = []
for _, row in tqdm(eval_gen_df.iterrows(), total=len(eval_gen_df)):
    generated = generate_one(ft_model, row["description"])
    if generation_has_artifacts(generated):
        print("WARNING: artifact-like generation detected for", row["source_file"])
    ft_rows.append({
        "source_story_id": row["story_id"],
        "source_file": row["source_file"],
        "description": row["description"],
        "text": generated,
        "label": "ai_finetuned",
        "generator": CONFIG["GEN_MODEL_NAME"] + " + QLoRA",
    })

ft_ai_df = pd.DataFrame(ft_rows)
ft_path = DATA_DIR / "ai_finetuned_generations.csv"
ft_ai_df.to_csv(ft_path, index=False)

print("Первые fine-tuned генерации. Здесь НЕ должно быть GraphicsUnit/#echo/случайных языков.")
display(ft_ai_df[["source_file", "text"]].head(5))
print("Сохранено:", ft_path)


In [ ]:
#@title 12. Стилометрическое сравнение
CLICHE_PATTERNS = [
    "shiver down my spine", "chill down my spine", "blood ran cold",
    "couldn't shake the feeling", "could not shake the feeling",
    "darkness seemed alive", "the darkness seemed", "heart pounded",
    "heart hammered", "air grew thick", "heavy with the scent",
    "like living things", "unseen presence", "whispered my name",
]

def cliche_rate(text: str) -> float:
    lower = str(text).lower()
    return sum(1 for pattern in CLICHE_PATTERNS if pattern in lower)

FEAR_WORDS = {
    "fear", "afraid", "scared", "terror", "horror", "dark", "darkness", "blood", "dead", "death",
    "scream", "screaming", "shadow", "shadows", "night", "nightmare", "ghost", "monster", "creature",
    "door", "window", "whisper", "whispers", "cold", "alone", "body", "grave", "madness", "evil",
}


def extract_features(text: str) -> dict[str, float]:
    words = re.findall(r"[A-Za-z][A-Za-z']*", str(text).lower())
    chars = len(str(text))
    unique_words = len(set(words))
    word_count = len(words)
    return {
        "n_chars": chars,
        "n_words": word_count,
        "type_token_ratio": unique_words / max(1, word_count),
        "avg_word_len": np.mean([len(word) for word in words]) if words else 0,
        "exclamation_count": str(text).count("!"),
        "question_count": str(text).count("?"),
        "ellipsis_count": str(text).count("..."),
        "quote_count": str(text).count('"') + str(text).count("“") + str(text).count("”"),
        "fear_word_rate": sum(1 for word in words if word in FEAR_WORDS) / max(1, word_count),
        "cliche_count": cliche_rate(text),
    }

human_eval_df = eval_gen_df[["text"]].copy()
human_eval_df["label"] = "human"
analysis_df = pd.concat([
    human_eval_df,
    base_ai_df[["text", "label"]],
    ft_ai_df[["text", "label"]],
], ignore_index=True)
features_df = pd.concat([analysis_df, analysis_df["text"].apply(extract_features).apply(pd.Series)], axis=1)
features_path = OUT_DIR / "stylometric_features.csv"
features_df.to_csv(features_path, index=False)
display(features_df.groupby("label").mean(numeric_only=True).round(3))
print("Сохранено:", features_path)


In [ ]:
#@title 13. Размеры эффекта: rank-biserial correlation
from scipy.stats import mannwhitneyu

feature_cols = [
    "n_chars", "n_words", "type_token_ratio", "avg_word_len", "exclamation_count",
    "question_count", "ellipsis_count", "quote_count", "fear_word_rate", "cliche_count",
]

rows = []
for ai_label in ["ai_base", "ai_finetuned"]:
    for feature in feature_cols:
        human_values = features_df.loc[features_df["label"] == "human", feature].to_numpy()
        ai_values = features_df.loc[features_df["label"] == ai_label, feature].to_numpy()
        u_stat, p_value = mannwhitneyu(human_values, ai_values, alternative="two-sided")
        n1, n2 = len(human_values), len(ai_values)
        rank_biserial = (2 * u_stat) / (n1 * n2) - 1
        rows.append({
            "comparison": f"human_vs_{ai_label}",
            "feature": feature,
            "rank_biserial": rank_biserial,
            "abs_effect": abs(rank_biserial),
            "p_value": p_value,
        })

effects_df = pd.DataFrame(rows).sort_values(["comparison", "abs_effect"], ascending=[True, False])
effects_path = OUT_DIR / "rank_biserial_effects.csv"
effects_df.to_csv(effects_path, index=False)
display(effects_df)
print("Сохранено:", effects_path)


In [ ]:
#@title 14. Supervised detector: human vs base и human vs fine-tuned
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments


def make_detector_df(human_texts, ai_texts, ai_name):
    human = pd.DataFrame({"text": human_texts, "label": 0, "class_name": "human"})
    ai = pd.DataFrame({"text": ai_texts, "label": 1, "class_name": ai_name})
    return pd.concat([human, ai], ignore_index=True).sample(frac=1, random_state=CONFIG["SEED"]).reset_index(drop=True)


def train_detector(detector_df, run_name):
    train_part, test_part = train_test_split(
        detector_df,
        test_size=0.3,
        random_state=CONFIG["SEED"],
        stratify=detector_df["label"],
    )
    det_tokenizer = AutoTokenizer.from_pretrained(CONFIG["DETECTOR_MODEL_NAME"])
    det_model = AutoModelForSequenceClassification.from_pretrained(CONFIG["DETECTOR_MODEL_NAME"], num_labels=2)

    def tokenize(batch):
        return det_tokenizer(batch["text"], truncation=True, max_length=512)

    train_ds = Dataset.from_pandas(train_part[["text", "label"]]).map(tokenize, batched=True)
    test_ds = Dataset.from_pandas(test_part[["text", "label"]]).map(tokenize, batched=True)
    collator = DataCollatorWithPadding(tokenizer=det_tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
        precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
        metrics = {
            "accuracy": accuracy_score(labels, preds),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        try:
            metrics["roc_auc"] = roc_auc_score(labels, probs)
        except Exception:
            metrics["roc_auc"] = float("nan")
        return metrics

    args = TrainingArguments(
        output_dir=str(MODEL_DIR / f"detector_{run_name}"),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=1 if CONFIG["SMOKE_TEST"] else 3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        report_to="none",
    )
    detector_trainer = Trainer(
        model=det_model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        processing_class=det_tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    detector_trainer.train()
    return detector_trainer.evaluate()

human_texts = eval_gen_df["text"].tolist()
base_detector_df = make_detector_df(human_texts, base_ai_df["text"].tolist(), "ai_base")
ft_detector_df = make_detector_df(human_texts, ft_ai_df["text"].tolist(), "ai_finetuned")

base_metrics = train_detector(base_detector_df, "human_vs_base_ai")
ft_metrics = train_detector(ft_detector_df, "human_vs_finetuned_ai")

summary = pd.DataFrame([
    {"scenario": "human vs base AI", **{k.replace("eval_", ""): v for k, v in base_metrics.items() if k.startswith("eval_")}},
    {"scenario": "human vs fine-tuned AI", **{k.replace("eval_", ""): v for k, v in ft_metrics.items() if k.startswith("eval_")}},
])
summary_path = OUT_DIR / "detector_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)
print("Сохранено:", summary_path)


In [ ]:
#@title 15. Экспорт результатов
zip_out = "/content/horror_experiment_results"
shutil.make_archive(zip_out, "zip", WORK_DIR)
print("Архив создан:", zip_out + ".zip")
files.download(zip_out + ".zip")
